# Graduate retention and brain drain — YNYCA in national context

Exploratory analysis using the ONS Graduate Retention dataset, which tracks a GCSE
cohort from 2008–11 to 2018–19 for 1,104 English built-up areas. Three headline
metrics per town:

- **Graduate retention rate** — share of local graduates who stayed in or returned to the town
- **Graduate inward-migration rate** — graduates from elsewhere who moved in
- **Graduate net gain rate** — the brain drain/gain balance: positive = net attractor of graduates, negative = net loser

Data also includes a parallel sheet for Level 3–5 non-graduates (sub-degree qualified),
enabling comparison of whether graduate loss is a specific phenomenon or reflects
broader population flows.


## 1. Imports and paths

In [ ]:
import pandas as pd
import numpy as np
import altair as alt
from ecostyles import EcoStyles
styles = EcoStyles()
styles.register_and_enable_theme(theme_name="article")
from scipy.stats import pearsonr

alt.data_transformers.enable("default", max_rows=None)
alt.renderers.enable("mimetype")  # use VSCode built-in Vega renderer, avoids requirejs CDN errors

BASE = "/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Documents/GitHub/RADataHub/msa_map"
DATA_PATH = f"{BASE}/data/raw/ons_grad_retention_data.xlsx"

TEAL       = "#2A9D8F"
NAVY       = "#1F3864"
BACKGROUND = "#F5F4EF"

# YNYCA built-up area TTWAs (York + North Yorkshire unitary districts)
YNYCA_TTWAS = {"York", "Harrogate", "Northallerton", "Skipton", "Malton", "Scarborough", "Whitby"}


## 2. Load data

In [2]:
# ── Graduates sheet ────────────────────────────────────────────────────────
grad_raw = pd.read_excel(DATA_PATH, sheet_name="Graduates", header=0)

# Suppressed values are marked as '*' — convert to NaN throughout
NUMERIC_COLS = [
    "GCSE Cohort 2008 to 2011", "Number of graduates", "Proportion of graduates",
    "Share of graduates who moved for HE and stayed afterwards",
    "Share of graduates who moved for HE and returned afterwads",
    "Share of graduates who moved for HE and moved elsewhere afterwards",
    "Share of graduates who stayed during HE but moved elsewhere afterwards",
    "Share of graduates who remained in town or city during and afterwards",
    "Share of graduates who moved town or city by 2018-19 (movers)",
    "Share of graduates who moved TTWA for HE ",
    "Share of graduate movers who moved TTWA by 2018-19",
    "Share of graduate movers who moved town or city but remained in the same TTWA by 2018-19",
    "Share of graduate movers who moved region by 2018-19",
    "Graduate retention rate", "Graduate inward-migration rate", "Graduate net gain rate",
    "Population_2011", "Level 4_or above_residents_age35to64_2011", "Education Score",
]
for col in NUMERIC_COLS:
    if col in grad_raw.columns:
        grad_raw[col] = pd.to_numeric(grad_raw[col], errors="coerce")

# Clean column names for convenience
grad = grad_raw.rename(columns={
    "Town and City Code":       "town_code",
    "Town and City Name":       "town_name",
    "GCSE Cohort 2008 to 2011": "gcse_cohort",
    "Number of graduates":      "n_graduates",
    "Proportion of graduates":  "pct_graduates",
    "Graduate retention rate":  "retention_rate",
    "Graduate inward-migration rate": "inward_rate",
    "Graduate net gain rate":   "net_gain_rate",
    "Population_2011":          "population",
    "Area_type":                "area_type",
    "Region":                   "region",
    "TTWA name":                "ttwa_name",
    "University flag":          "university_flag",
    "Income deprivation flag":  "deprivation_flag",
    "Education Score":          "education_score",
    "Level 4_or above_residents_age35to64_2011": "pct_level4_2011",
})

# ── Non-graduates sheet ────────────────────────────────────────────────────
nong_raw = pd.read_excel(DATA_PATH, sheet_name="L3 to L5 non-graduates", header=0)
for col in nong_raw.columns:
    if col != "Town and City Name":
        nong_raw[col] = pd.to_numeric(nong_raw[col], errors="coerce")
nong = nong_raw.rename(columns={
    "Town and City Name": "town_name",
    "L3 to L5 non-graduates retention rate": "nong_retention_rate",
})[["town_name", "nong_retention_rate"]]

# Merge retention comparison onto main frame
grad = grad.merge(nong, on="town_name", how="left")

print(f"Total towns in dataset: {len(grad)}")
print(f"Towns with non-suppressed net gain rate: {grad['net_gain_rate'].notna().sum()}")
print(f"Regions: {sorted(grad['region'].dropna().unique())}")


Total towns in dataset: 1104
Towns with non-suppressed net gain rate: 933
Regions: ['East Midlands', 'East of England', 'London', 'North East', 'North West', 'South East', 'South West', 'West Midlands', 'Yorkshire and The Humber', 'Yorkshire and the Humber']


## 3. National descriptive statistics

In [3]:
print("=== Key metric distributions — all English towns ===")
desc_cols = ["retention_rate", "inward_rate", "net_gain_rate", "pct_graduates"]
print(grad[desc_cols].describe().round(3))

print()
print("=== Net gain rate distribution ===")
print(f"  Brain gain  (>  0): {(grad['net_gain_rate'] >  0).sum()} towns  ({(grad['net_gain_rate'] >  0).mean()*100:.0f}%)")
print(f"  Brain drain (<  0): {(grad['net_gain_rate'] <  0).sum()} towns  ({(grad['net_gain_rate'] <  0).mean()*100:.0f}%)")
print(f"  Suppressed  (*): {grad['net_gain_rate'].isna().sum()} towns")
print()
print(f"National median net gain rate: {grad['net_gain_rate'].median():.3f}")
print(f"National median retention rate: {grad['retention_rate'].median():.3f}")

print()
print("=== University premium — median net gain rate ===")
print(grad.groupby("university_flag")["net_gain_rate"].agg(["median","count"]).round(3))


=== Key metric distributions — all English towns ===
       retention_rate  inward_rate  net_gain_rate  pct_graduates
count        1066.000      933.000        933.000       1104.000
mean            0.550        0.098         -0.125          0.316
std             0.088        0.066          0.233          0.098
min             0.294        0.021         -0.561          0.087
25%             0.487        0.060         -0.254          0.244
50%             0.553        0.082         -0.170          0.300
75%             0.611        0.110         -0.067          0.376
max             0.818        0.614          2.023          0.722

=== Net gain rate distribution ===
  Brain gain  (>  0): 136 towns  (12%)
  Brain drain (<  0): 777 towns  (70%)
  Suppressed  (*): 171 towns

National median net gain rate: -0.170
National median retention rate: 0.553

=== University premium — median net gain rate ===
                 median  count
university_flag               
No University    -0.181    86

### National distribution of graduate net gain rate

Distribution across all 1,104 English built-up areas — shows how most towns are net
losers of graduates, with a long right tail of attractor cities.


In [4]:
# Histogram of net gain rate nationally
hist_data = grad[grad['net_gain_rate'].notna()].copy()

histogram = alt.Chart(hist_data).mark_bar(color=TEAL, opacity=0.8).encode(
    x=alt.X("net_gain_rate:Q", bin=alt.Bin(step=0.1),
             title="Graduate net gain rate (negative = brain drain, positive = gain)"),
    y=alt.Y("count()", title="Number of towns"),
    tooltip=[alt.Tooltip("count()", title="Towns")],
).properties(
    width=500, height=240,
    title=alt.TitleParams(
        text="Distribution of graduate net gain rate — all English built-up areas",
        subtitle="Negative = more graduates leave than arrive. Most towns are net losers.",
        fontSize=13, subtitleFontSize=11, anchor="start",
    ),
).configure_view(strokeWidth=0).configure(background=BACKGROUND).configure_axis(grid=False)

histogram


<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


## 4. YNYCA towns — filter and profile

In [5]:
ynyca = grad[grad["ttwa_name"].isin(YNYCA_TTWAS)].copy()
ynyca["is_ynyca"] = True

print(f"YNYCA built-up areas: {len(ynyca)}")
print(f"With usable net gain rate: {ynyca['net_gain_rate'].notna().sum()}")
print()

profile_cols = ["town_name", "area_type", "university_flag",
                "retention_rate", "inward_rate", "net_gain_rate",
                "nong_retention_rate", "ttwa_name"]
print(ynyca[profile_cols].sort_values("net_gain_rate", ascending=False).to_string(index=False))


YNYCA built-up areas: 22
With usable net gain rate: 19

              town_name area_type university_flag  retention_rate  inward_rate  net_gain_rate  nong_retention_rate     ttwa_name
             York BUASD     Large      University           0.610        0.447          1.034                0.766          York
            Selby BUASD     Small   No University           0.500        0.117          0.059                0.620          York
Norton-on-Derwent BUASD     Small   No University           0.533        0.100         -0.067                0.636        Malton
             Thirsk BUA     Small   No University           0.519        0.121         -0.074                0.667 Northallerton
  Sherburn in Elmet BUA     Small   No University           0.478        0.135         -0.087                0.609          York
            Skipton BUA     Small   No University           0.558        0.120         -0.115                0.673       Skipton
        Pocklington BUA     Small   No Un

### How YNYCA towns compare to the national distribution

In [6]:
# Context: YNYCA vs national benchmarks
ynyca_valid = ynyca[ynyca["net_gain_rate"].notna()]
national_valid = grad[grad["net_gain_rate"].notna()]

print("=== YNYCA vs national benchmarks ===")
for metric, label in [("net_gain_rate","Net gain rate"), ("retention_rate","Retention rate"),
                       ("inward_rate","Inward migration rate")]:
    nat_med = national_valid[metric].median()
    ynyca_med = ynyca_valid[metric].median()
    n_above = (ynyca_valid[metric] > nat_med).sum()
    n_total = ynyca_valid[metric].notna().sum()
    print(f"  {label}:")
    print(f"    National median: {nat_med:+.3f}   YNYCA median: {ynyca_med:+.3f}")
    print(f"    YNYCA towns above national median: {n_above}/{n_total}")
    print()

print(f"Brain gain towns in YNYCA:  {(ynyca['net_gain_rate'] > 0).sum()}")
print(f"Brain drain towns in YNYCA: {(ynyca['net_gain_rate'] < 0).sum()}")
print(f"Suppressed:                 {ynyca['net_gain_rate'].isna().sum()}")


=== YNYCA vs national benchmarks ===
  Net gain rate:
    National median: -0.170   YNYCA median: -0.286
    YNYCA towns above national median: 7/19

  Retention rate:
    National median: +0.560   YNYCA median: +0.490
    YNYCA towns above national median: 1/19

  Inward migration rate:
    National median: +0.082   YNYCA median: +0.093
    YNYCA towns above national median: 10/19

Brain gain towns in YNYCA:  2
Brain drain towns in YNYCA: 17
Suppressed:                 3


### Graduate vs non-graduate retention — are graduates specifically more likely to leave?

In [7]:
compare = ynyca[["town_name","retention_rate","nong_retention_rate","net_gain_rate"]].dropna(
    subset=["retention_rate","nong_retention_rate"]
).copy()
compare["grad_minus_nong"] = compare["retention_rate"] - compare["nong_retention_rate"]

print("Retention rate: graduates vs non-graduates")
print(compare[["town_name","retention_rate","nong_retention_rate","grad_minus_nong"]].sort_values(
    "grad_minus_nong").to_string(index=False))
print()
print(f"Towns where graduates retain LESS than non-graduates: {(compare['grad_minus_nong'] < 0).sum()}")
print(f"Towns where graduates retain MORE than non-graduates: {(compare['grad_minus_nong'] > 0).sum()}")
print(f"National picture: median grad retention {grad['retention_rate'].median():.3f}, "
      f"median non-grad retention {grad['nong_retention_rate'].median():.3f}")


Retention rate: graduates vs non-graduates
              town_name  retention_rate  nong_retention_rate  grad_minus_nong
           Richmond BUA           0.341                0.600           -0.259
        Pickering BUASD           0.409                0.667           -0.258
             Whitby BUA           0.464                0.722           -0.258
              Ripon BUA           0.447                0.667           -0.220
        Harrogate BUASD           0.543                0.758           -0.215
        Scarborough BUA           0.541                0.747           -0.206
          Tadcaster BUA           0.520                0.720           -0.200
             York BUASD           0.610                0.766           -0.156
             Thirsk BUA           0.519                0.667           -0.148
              Haxby BUA           0.463                0.607           -0.144
          Strensall BUA           0.429                0.571           -0.142
      Northallerton B

## 5. Save YNYCA subset for further analysis

In [8]:
ynyca.to_csv(f"{BASE}/data/processed/ynyca_grad_retention.csv", index=False)
print(f"Saved {len(ynyca)} rows to data/processed/ynyca_grad_retention.csv")


Saved 22 rows to data/processed/ynyca_grad_retention.csv


## 6. True MSA-level brain drain — graduates confirmed leaving the region

The town-level net gain rates overstate YNYCA brain drain because they count
within-YNYCA moves as losses (e.g. Ripon→York). The dataset doesn't record
*where* movers went, so we can't identify what share went to York (internal
redistribution, no MSA-level loss) vs. Leeds (outside YNYCA, still in Yorkshire)
vs. London (outside the region entirely).

We can compute a **confirmed lower bound on true MSA brain drain**:

> share who moved away from their town × share of those movers who left
> Yorkshire and The Humber entirely

This is a floor — some movers who stayed within Yorkshire will also have left YNYCA
(e.g. going to Leeds), so the true MSA-level outflow is higher, but we can't
quantify it precisely without destination data.


In [9]:
MOVERS_COL   = "Share of graduates who moved town or city by 2018-19 (movers)"
LEFT_REG_COL = "Share of graduate movers who moved region by 2018-19"
SAME_TTWA    = "Share of graduate movers who moved town or city but remained in the same TTWA by 2018-19"

ynyca["pct_left_region"] = ynyca[MOVERS_COL] * pd.to_numeric(ynyca[LEFT_REG_COL], errors="coerce")
ynyca["pct_same_ttwa"]   = ynyca[MOVERS_COL] * pd.to_numeric(ynyca[SAME_TTWA], errors="coerce").fillna(0)

nat_left_region = (
    pd.to_numeric(grad_raw[MOVERS_COL], errors="coerce") *
    pd.to_numeric(grad_raw[LEFT_REG_COL], errors="coerce")
).median()

flow = ynyca[["town_name", "ttwa_name", MOVERS_COL, LEFT_REG_COL,
              "pct_left_region", "pct_same_ttwa"]].copy()
flow.columns = ["town", "ttwa", "pct_movers", "of_movers_left_region",
                "pct_all_left_region", "pct_all_same_ttwa"]
flow = flow.dropna(subset=["pct_all_left_region"]).sort_values("pct_all_left_region", ascending=False)

print("=== Confirmed out-of-region exits as share of ALL graduates ===")
print(flow.round(3).to_string(index=False))
print()
print(f"National median (confirmed regional exit): {nat_left_region:.3f}")
print(f"YNYCA median:                              {flow['pct_all_left_region'].median():.3f}")


=== Confirmed out-of-region exits as share of ALL graduates ===
                  town          ttwa  pct_movers  of_movers_left_region  pct_all_left_region  pct_all_same_ttwa
Catterick Garrison BUA Northallerton       0.667                  0.800                0.534              0.000
          Richmond BUA Northallerton       0.634                  0.692                0.439              0.000
            Thirsk BUA Northallerton       0.444                  0.750                0.333              0.000
            Whitby BUA        Whitby       0.500                  0.643                0.322              0.000
       Harrogate BUASD     Harrogate       0.430                  0.693                0.298              0.041
             Ripon BUA     Harrogate       0.532                  0.560                0.298              0.000
     Northallerton BUA Northallerton       0.469                  0.609                0.286              0.000
        Wetherby BUASD     Harrogate    

In [10]:
flow["category"] = flow["town"].apply(
    lambda t: "Coastal district" if "Scarborough" in t
    else ("City" if any(c in t for c in ["York BUASD","Ripon"]) else "Market town")
)

COLOR_SCALE = alt.Scale(
    domain=["Coastal district", "City", "Market town"],
    range=[TEAL, NAVY, "#cccccc"],
)

bars = alt.Chart(flow).mark_bar(cornerRadiusEnd=3).encode(
    x=alt.X("pct_all_left_region:Q",
             title="Share of all graduates confirmed leaving the Yorkshire region",
             scale=alt.Scale(domain=[0, 0.6]),
             axis=alt.Axis(labelExpr="round(datum.value*100) + '%'", grid=False)),
    y=alt.Y("town:N", sort=alt.SortField("pct_all_left_region", "descending"),
             title=None, axis=alt.Axis(grid=False)),
    color=alt.Color("category:N", scale=COLOR_SCALE,
                    legend=alt.Legend(title="Town type", orient="bottom")),
    tooltip=[
        alt.Tooltip("town:N", title="Town"),
        alt.Tooltip("pct_movers:Q", title="% who moved at all", format=".1%"),
        alt.Tooltip("of_movers_left_region:Q", title="Of movers, % left region", format=".1%"),
        alt.Tooltip("pct_all_left_region:Q", title="All grads: % left region", format=".1%"),
        alt.Tooltip("pct_all_same_ttwa:Q", title="All grads: % moved, same TTWA", format=".1%"),
    ],
)

ref = alt.Chart(pd.DataFrame({"x": [nat_left_region]})).mark_rule(
    color=NAVY, strokeDash=[4, 3], opacity=0.5
).encode(x="x:Q")

ref_label = alt.Chart(pd.DataFrame({"x": [nat_left_region], "label": ["National baseline"]})).mark_text(
    align="left", dx=4, color=NAVY, fontSize=9, fontStyle="italic",
).encode(x="x:Q", y=alt.value(8), text="label:N")

chart_regional = (bars + ref + ref_label).properties(
    width=380, height=320,
    title=alt.TitleParams(
        text="Graduates confirmed leaving the Yorkshire region — YNYCA towns",
        subtitle=[
            "Lower bound on true YNYCA-level brain drain (some within-Yorkshire movers also left YNYCA).",
            "Dashed line = national median. Source: ONS Graduate Retention Study.",
        ],
        fontSize=13, subtitleFontSize=10, anchor="start",
    ),
).configure_view(strokeWidth=0).configure(background=BACKGROUND).configure_axis(grid=False)

chart_regional


<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


### Key findings

- **YNYCA's median confirmed regional exit rate (27%) sits modestly above the national
  median (24%)**, confirming the MSA is a net exporter of graduates at the regional
  level — the picture isn't just internal redistribution toward York.

- **Catterick Garrison (53%) and Richmond (44%)** are extreme outliers — both in the
  Northallerton/Richmondshire area. Catterick's military character means trained people
  routinely move nationally on posting. Richmond has very limited local economic pull.

- **Selby (18%)** is the lowest — its proximity to Leeds means many who leave Selby
  likely go to West Yorkshire (outside YNYCA, but within Yorkshire), so the regional
  exit figure *understates* its true MSA-level loss.

- **York (26%)** shows a high regional exit rate despite being the MSA's main
  attractor — graduates who *do* leave York tend to go to London or other major cities,
  not nearby towns, reflecting its position as a national rather than purely local
  labour market.

- The within-TTWA stayer rate is only material for York-TTWA settlements (Haxby,
  Strensall, Selby: 15–18%), confirming these smaller settlements channel graduates
  toward York itself rather than exporting them out of the MSA.


## 7. Graduate vs non-graduate retention — dumbbell chart

Each town shown as a pair of dots (connected by a line): left dot = graduate
retention rate, right dot = non-graduate (Level 3–5) retention rate. The gap
between the two dots shows how much *more* likely non-graduates are to stay.
Reference lines show the England-wide median and Yorkshire & the Humber median
for each group.


In [11]:
# ── Reference averages ────────────────────────────────────────────────────
ENG_GRAD  = grad_raw["Graduate retention rate"].apply(pd.to_numeric, errors="coerce").median()
ENG_NONG  = pd.to_numeric(
    pd.read_excel(DATA_PATH, sheet_name="L3 to L5 non-graduates", header=0)
    .rename(columns={"L3 to L5 non-graduates retention rate": "nong"})[["Town and City Name","nong"]]
    .merge(grad_raw[["Town and City Name"]], on="Town and City Name")["nong"],
    errors="coerce",
).median()

yorks_mask = grad_raw["Region"].str.contains("Yorkshire", na=False)
YORK_GRAD = pd.to_numeric(grad_raw.loc[yorks_mask, "Graduate retention rate"], errors="coerce").median()
# (non-grad Yorkshire avg: use the merged frame available in ynyca)
YORK_NONG = pd.to_numeric(
    pd.read_excel(DATA_PATH, sheet_name="L3 to L5 non-graduates", header=0)
    .rename(columns={"Town and City Name":"town_name","L3 to L5 non-graduates retention rate":"nong"})
    [["town_name","nong"]]
    .merge(grad_raw.rename(columns={"Town and City Name":"town_name"})[["town_name","Region"]], on="town_name")
    .query("Region.str.contains('Yorkshire', na=False)")["nong"],
    errors="coerce",
).median()

print(f"England  — grad: {ENG_GRAD:.3f}  non-grad: {ENG_NONG:.3f}")
print(f"Yorkshire — grad: {YORK_GRAD:.3f}  non-grad: {YORK_NONG:.3f}")


England  — grad: 0.553  non-grad: 0.638
Yorkshire — grad: 0.541  non-grad: 0.633


In [12]:
# ── Build long-format data for dumbbell ───────────────────────────────────
dumbbell_df = ynyca[["town_name", "retention_rate", "nong_retention_rate"]].copy()
dumbbell_df = dumbbell_df.dropna(subset=["retention_rate", "nong_retention_rate"])
dumbbell_df = dumbbell_df.sort_values("retention_rate")

melted = dumbbell_df.melt(
    id_vars="town_name",
    value_vars=["retention_rate", "nong_retention_rate"],
    var_name="metric",
    value_name="rate",
)
melted["metric_label"] = melted["metric"].map({
    "retention_rate": "Graduates",
    "nong_retention_rate": "Non-graduates (L3–5)",
})

town_order = dumbbell_df["town_name"].tolist()

GRAD_COLOR = NAVY
NONG_COLOR = "#A8C5BD"  # muted teal — distinct from navy but in palette

# Reference lines: 4 rules (England grad, England non-grad, Yorkshire grad, Yorkshire non-grad)
refs = pd.DataFrame([
    {"x": ENG_GRAD,  "label": "England avg (graduates)",     "color": GRAD_COLOR, "dash": [4, 3]},
    {"x": ENG_NONG,  "label": "England avg (non-graduates)",  "color": NONG_COLOR, "dash": [4, 3]},
    {"x": YORK_GRAD, "label": "Yorkshire avg (graduates)",    "color": GRAD_COLOR, "dash": [2, 2]},
    {"x": YORK_NONG, "label": "Yorkshire avg (non-graduates)","color": NONG_COLOR, "dash": [2, 2]},
])

# Connecting lines (one per town)
lines = alt.Chart(melted).mark_line(color="#dddddd", strokeWidth=1.5).encode(
    y=alt.Y("town_name:N", sort=town_order, title=None),
    x=alt.X("rate:Q"),
    detail="town_name:N",
)

# Dots
dots = alt.Chart(melted).mark_circle(size=80, opacity=0.9).encode(
    y=alt.Y("town_name:N", sort=town_order, title=None,
             axis=alt.Axis(labelFontSize=11, grid=False)),
    x=alt.X("rate:Q",
             title="Retention rate",
             scale=alt.Scale(domain=[0.28, 0.82]),
             axis=alt.Axis(labelExpr="round(datum.value*100) + '%'", grid=False)),
    color=alt.Color("metric_label:N",
                    scale=alt.Scale(domain=["Graduates", "Non-graduates (L3–5)"],
                                    range=[GRAD_COLOR, NONG_COLOR]),
                    legend=alt.Legend(title="Group", orient="bottom")),
    tooltip=[
        alt.Tooltip("town_name:N", title="Town"),
        alt.Tooltip("metric_label:N", title="Group"),
        alt.Tooltip("rate:Q", title="Retention rate", format=".1%"),
    ],
)

# Reference rules (drawn one by one so colours/dashes differ)
ref_layers = []
for _, row in refs.iterrows():
    ref_layers.append(
        alt.Chart(pd.DataFrame({"x": [row["x"]]})).mark_rule(
            color=row["color"], strokeDash=row["dash"], opacity=0.55, strokeWidth=1.2
        ).encode(x="x:Q")
    )

chart_dumbbell = alt.layer(lines, dots, *ref_layers).properties(
    width=420, height=380,
    title=alt.TitleParams(
        text="Graduate vs non-graduate retention — YNYCA towns",
        subtitle=[
            "Left dot = graduate retention rate.  Right dot = non-graduate (L3–5) retention rate.",
            "Dashed lines = England median.  Dotted lines = Yorkshire & the Humber median.",
            "Sorted by graduate retention rate (ascending).",
        ],
        fontSize=13, subtitleFontSize=10, anchor="start",
    ),
).configure_view(strokeWidth=0).configure(background=BACKGROUND).configure_axis(grid=False)

chart_dumbbell


<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


## 8. Graduate vs non-graduate retention — scatter

Each dot is a YNYCA town. X-axis = graduate retention rate;
Y-axis = non-graduate retention rate. Points above the dashed
diagonal line (y = x) stay closer to home than their graduate
peers. Selected towns labelled. Crosshair = England median.


## 9. Brain drain by town — graduate net gain rate

Diverging bar chart showing the net gain rate for each YNYCA town.
Positive = brain gain (more graduates arriving than leaving); negative = brain drain.
National median reference line shows where most English towns sit.


In [ ]:
# ── Net gain rate — diverging horizontal bar ──────────────────────────────
bar_df = ynyca[["town_name", "net_gain_rate", "university_flag"]].dropna(
    subset=["net_gain_rate"]
).copy()
bar_df["label"] = bar_df["town_name"].str.replace(r" BUA(?:SD)?$", "", regex=True)
bar_df["polarity"] = bar_df["net_gain_rate"].apply(
    lambda v: "Brain gain" if v > 0 else "Brain drain"
)
bar_df = bar_df.sort_values("net_gain_rate", ascending=False)
town_order = bar_df["label"].tolist()

NAT_MEDIAN = grad["net_gain_rate"].median()  # −0.170

# ── Bars ──────────────────────────────────────────────────────────────────
bars = alt.Chart(bar_df).mark_bar(cornerRadiusEnd=3, height=14).encode(
    x=alt.X(
        "net_gain_rate:Q",
        title="Graduate net gain rate",
        axis=alt.Axis(
            labelExpr="datum.value > 0 ? '+' + round(datum.value * 100) + '%' : round(datum.value * 100) + '%'",
            grid=False,
            tickCount=7,
        ),
    ),
    y=alt.Y("label:N", sort=town_order, title=None,
             axis=alt.Axis(grid=False, labelFontSize=11)),
    color=alt.Color(
        "polarity:N",
        scale=alt.Scale(domain=["Brain gain", "Brain drain"], range=[TEAL, "#C0392B"]),
        legend=alt.Legend(title=None, orient="bottom", symbolType="square"),
    ),
    tooltip=[
        alt.Tooltip("label:N", title="Town"),
        alt.Tooltip("net_gain_rate:Q", title="Net gain rate",
                    format="+.1%"),
        alt.Tooltip("university_flag:N", title="University"),
    ],
)

# ── Zero baseline ─────────────────────────────────────────────────────────
zero_rule = alt.Chart(pd.DataFrame({"x": [0]})).mark_rule(
    color="#333333", strokeWidth=1.5, opacity=0.85
).encode(x="x:Q")

# ── National median reference ─────────────────────────────────────────────
nat_rule = alt.Chart(pd.DataFrame({"x": [NAT_MEDIAN]})).mark_rule(
    color=NAVY, strokeDash=[5, 3], opacity=0.55, strokeWidth=1.2
).encode(x="x:Q")

nat_label = alt.Chart(
    pd.DataFrame({"x": [NAT_MEDIAN], "label": [f"National median ({round(NAT_MEDIAN*100)}%)"]})
).mark_text(
    align="right", dx=-5, fontSize=9, fontStyle="italic", color=NAVY,
).encode(x="x:Q", y=alt.value(16), text="label:N")

# ── Compose ───────────────────────────────────────────────────────────────
chart_brain_drain = (bars + zero_rule + nat_rule + nat_label).properties(
    width=480, height=400,
    title=alt.TitleParams(
        text="Brain drain by town — YNYCA graduate net gain rate",
        subtitle=[
            "Net gain rate = graduates arriving minus graduates leaving, as share of local cohort.",
            "17 of 19 YNYCA towns are net losers of graduates. Dashed line = national median (−17%).",
            "York's university status drives MSA-wide inflow. Source: ONS Graduate Retention Study.",
        ],
        fontSize=13, subtitleFontSize=10, anchor="start",
    ),
).configure_view(strokeWidth=0).configure(background=BACKGROUND).configure_axis(grid=False)

chart_brain_drain


In [13]:
scatter_df = ynyca[["town_name", "retention_rate", "nong_retention_rate",
                     "net_gain_rate", "university_flag"]].dropna(
    subset=["retention_rate", "nong_retention_rate"]
).copy()

# Clean display names
scatter_df["label"] = scatter_df["town_name"].str.replace(r" BUA(?:SD)?$", "", regex=True)

# Town-type colour (consistent with earlier charts)
COASTAL = {"Scarborough"}
CITIES  = {"York", "Ripon"}
scatter_df["category"] = scatter_df["label"].apply(
    lambda t: "Coastal" if t in COASTAL else ("City" if t in CITIES else "Market town")
)

# Towns to label (select the most policy-relevant / analytically interesting)
LABEL_TOWNS = {
    "York", "Scarborough", "Richmond", "Ripon",
    "Whitby", "Harrogate", "Catterick Garrison",
}

COLOR_SCALE = alt.Scale(
    domain=["Coastal", "City", "Market town"],
    range=[TEAL, NAVY, "#aaaaaa"],
)

# ── Diagonal reference (y = x, equal retention for both groups) ───────────
axis_range = [0.25, 0.85]
diag = alt.Chart(pd.DataFrame({"x": axis_range, "y": axis_range})).mark_line(
    color="#999999", strokeDash=[5, 4], opacity=0.6
).encode(x="x:Q", y="y:Q")

# ── England median crosshair ──────────────────────────────────────────────
eng_point = pd.DataFrame([{"x": ENG_GRAD, "y": ENG_NONG, "label": "England median"}])

cross_h = alt.Chart(eng_point).mark_rule(
    color=NAVY, strokeDash=[3, 3], opacity=0.35
).encode(y="y:Q")
cross_v = alt.Chart(eng_point).mark_rule(
    color=NAVY, strokeDash=[3, 3], opacity=0.35
).encode(x="x:Q")
cross_dot = alt.Chart(eng_point).mark_point(
    shape="cross", size=60, color=NAVY, opacity=0.5
).encode(x="x:Q", y="y:Q",
         tooltip=[alt.Tooltip("label:N")])

# ── Scatter dots ──────────────────────────────────────────────────────────
dots = alt.Chart(scatter_df).mark_circle(size=75, opacity=0.85).encode(
    x=alt.X("retention_rate:Q",
             title="Graduate retention rate",
             scale=alt.Scale(domain=[0.28, 0.65]),
             axis=alt.Axis(labelExpr="round(datum.value*100) + '%'", grid=False)),
    y=alt.Y("nong_retention_rate:Q",
             title="Non-graduate retention rate",
             scale=alt.Scale(domain=[0.25, 0.85]),
             axis=alt.Axis(labelExpr="round(datum.value*100) + '%'", grid=False)),
    color=alt.Color("category:N", scale=COLOR_SCALE,
                    legend=alt.Legend(title="Town type", orient="bottom")),
    tooltip=[
        alt.Tooltip("label:N", title="Town"),
        alt.Tooltip("retention_rate:Q", title="Grad retention", format=".1%"),
        alt.Tooltip("nong_retention_rate:Q", title="Non-grad retention", format=".1%"),
        alt.Tooltip("net_gain_rate:Q", title="Net gain rate", format="+.3f"),
    ],
)

# ── Text labels for selected towns ────────────────────────────────────────
labeled = scatter_df[scatter_df["label"].isin(LABEL_TOWNS)].copy()

# Manual nudges for key towns to avoid overlap
DX = {"York": 7, "Scarborough": 7, "Richmond": 7, "Ripon": 7,
      "Whitby": -8, "Harrogate": 7, "Catterick Garrison": 7}
DY = {"York": -10, "Scarborough": 0, "Richmond": 0, "Ripon": 0,
      "Whitby": 0, "Harrogate": 8, "Catterick Garrison": -10}
ALIGN = {"Whitby": "right"}
WEIGHT = {"York": "bold", "Scarborough": "bold", "Catterick Garrison": "bold"}

txt_layers = []
for _, row in labeled.iterrows():
    t = row["label"]
    txt_layers.append(
        alt.Chart(pd.DataFrame([row])).mark_text(
            align=ALIGN.get(t, "left"),
            dx=DX.get(t, 7), dy=DY.get(t, 0),
            fontSize=9.5,
            fontWeight=WEIGHT.get(t, "normal"),
            color=NAVY if t in CITIES else (TEAL if t in COASTAL else "#555555"),
        ).encode(
            x="retention_rate:Q",
            y="nong_retention_rate:Q",
            text="label:N",
        )
    )

chart_scatter = alt.layer(
    diag, cross_h, cross_v, cross_dot,
    dots, *txt_layers,
).properties(
    width=460, height=420,
    title=alt.TitleParams(
        text="Graduate vs non-graduate retention — YNYCA towns",
        subtitle=[
            "Points above the dashed line = non-graduates are retained at a higher rate than graduates.",
            "Catterick Garrison (below the line) is the sole exception — a military-base effect.",
            "Crosshair = England median. Source: ONS Graduate Retention Study.",
        ],
        fontSize=13, subtitleFontSize=10, anchor="start",
    ),
).configure_view(strokeWidth=0).configure(background=BACKGROUND).configure_axis(grid=False)

chart_scatter


<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting
